# HMS Codebook Theory & Benchmark Analysis

This notebook covers:
1. **Theoretical capacity bounds** for sparse binary (EntangledHVec) and Clifford algebra (CliffordVec)
2. **Empirical benchmarks** from the Rust stress test binary
3. **Similarity distribution** analysis and crosstalk characterization
4. **Retrieval quality** comparison: brute-force vs Hopfield-Fenchel-Young

In [ ]:
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy import special, stats
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

HMS_ROOT = Path('..').resolve()
print(f'HMS root: {HMS_ROOT}')

## 1. Run Stress Test

Build and run the Rust stress test binary with `--json` output.

In [ ]:
# Build the stress test binary (release mode for accurate timings)
build = subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'hms-stress'],
    cwd=HMS_ROOT, capture_output=True, text=True, timeout=600
)
if build.returncode != 0:
    print('Build failed:')
    print(build.stderr)
else:
    print('Build succeeded')

# Locate the binary (may be in a custom target-dir)
STRESS_BIN = None
for candidate in [
    HMS_ROOT / 'target/release/hms-stress',
    Path('/Volumes/C/rust-target/release/hms-stress'),
]:
    if candidate.exists():
        STRESS_BIN = str(candidate)
        break
assert STRESS_BIN, 'Could not find hms-stress binary'
print(f'Binary: {STRESS_BIN}')

In [ ]:
# Run the stress test
result = subprocess.run(
    [STRESS_BIN, '--json'],
    capture_output=True, text=True, timeout=300
)
if result.returncode != 0:
    print('Stress test failed:')
    print(result.stderr)
else:
    data = json.loads(result.stdout)
    print(f'Loaded results: {list(data.keys())}')

## 2. Theoretical Capacity Bounds

### Sparse Binary (EntangledHVec)
- Dimension $D = 16384$, sparsity $\rho = 1/256$, active bits $k = 64$
- Random pair similarity (Jaccard): $E[J] = \rho / (2 - \rho) \approx 0.00392$
- Capacity: number of patterns $N$ before crosstalk exceeds retrieval threshold

### Clifford Algebra (CliffordVec)
- $\text{Cl}(14, 0)$: algebra dimension $2^{14} = 16384$ blades
- Sparse multivector: 64 non-zero terms with $\pm 1$ coefficients
- Similarity via reverse inner product: $\text{sim}(a,b) = \langle a \cdot \tilde{b} \rangle_0 / (|a| \cdot |b|)$

In [ ]:
D = 16384
rho = 1 / 256
k = int(D * rho)  # 64 active bits

# --- Sparse Binary Capacity ---

# Expected Jaccard similarity for two random sparse binary vectors
# P(bit i in A AND B) = rho^2, P(bit i in A OR B) = 2*rho - rho^2
# E[|A∩B|] = D * rho^2, E[|A∪B|] = D * (2*rho - rho^2)
# E[Jaccard] = rho / (2 - rho)
expected_jaccard = rho / (2 - rho)
print(f'Dimension D = {D}')
print(f'Sparsity rho = 1/{int(1/rho)} = {rho:.6f}')
print(f'Active bits k = {k}')
print(f'Expected random Jaccard = {expected_jaccard:.6f}')
print()

# Variance of Jaccard for sparse binary
# Each bit is Bernoulli, so intersection count ~ Binomial(D, rho^2)
# Union count ~ Binomial(D, 2*rho - rho^2)
E_intersection = D * rho**2
E_union = D * (2*rho - rho**2)
print(f'Expected |A∩B| = {E_intersection:.2f}')
print(f'Expected |A∪B| = {E_union:.2f}')

# For N stored patterns, probability that at least one has Jaccard > threshold
# with the query by chance. Using normal approximation to binomial:
var_intersection = D * rho**2 * (1 - rho**2)
std_intersection = np.sqrt(var_intersection)
print(f'Std of |A∩B| = {std_intersection:.4f}')
print()

# Capacity estimation: N patterns where P(max noise > signal) < 0.01
# Signal: Jaccard(query, true_match) assuming perfect match ~ 1.0
# Noise floor: max of N-1 independent Jaccard values
# Using extreme value theory: max of N draws from Jaccard distribution
N_values = np.array([10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000])
noise_means = []
noise_maxes_99 = []

for N in N_values:
    # Approximate: intersection count ~ Normal(mu, sigma)
    # Max of N-1 such draws
    mu = E_intersection
    sigma = std_intersection
    # Expected max of N standard normals: ~ sigma * sqrt(2 * ln(N)) + mu
    if N > 1:
        expected_max_intersection = mu + sigma * np.sqrt(2 * np.log(N))
        # 99th percentile: add another ~0.5 * sigma * (ln(ln(N)) + ln(4*pi)) / sqrt(2*ln(N))
        max_jaccard = expected_max_intersection / E_union
    else:
        max_jaccard = expected_jaccard
    noise_means.append(max_jaccard)

print('Estimated noise floor (max random Jaccard):')
print(f'{"N":>8}  {"E[max Jaccard]":>15}')
print('-' * 30)
for n, nm in zip(N_values, noise_means):
    print(f'{n:>8}  {nm:>15.6f}')

In [ ]:
# --- Clifford Algebra Capacity ---

n_basis = 14  # Cl(14,0)
algebra_dim = 2**n_basis  # 16384 blades
n_terms = 64  # sparse terms per multivector

print(f'Clifford Cl({n_basis},0)')
print(f'Algebra dimension: {algebra_dim} blades')
print(f'Sparse terms: {n_terms}')
print()

# For CliffordVec with random +/-1 coefficients on k random blades:
# similarity = <a * b_rev>_0 / (|a| * |b|)
# Grade-0 contribution only when blade_a == blade_b
# P(shared blade) = k^2 / algebra_dim (birthday problem approximation)
# Expected shared blades = k^2 / algebra_dim
E_shared = n_terms**2 / algebra_dim
print(f'Expected shared blades: {E_shared:.4f}')

# Each shared blade contributes c_a * c_b to the inner product
# With +/-1 random coefficients, each contribution is +/-1 with equal probability
# So expected inner product = 0, variance = E_shared
# |a| = |b| = sqrt(k) = 8 for k=64
# Expected |similarity| = sqrt(E_shared) / k ≈ small
norm = np.sqrt(n_terms)  # sqrt(64) = 8
E_abs_sim = np.sqrt(E_shared) / (norm**2)
print(f'Expected |similarity|: {E_abs_sim:.6f}')
print(f'Norm |v| = sqrt({n_terms}) = {norm:.1f}')
print()

# Signal-to-noise ratio comparison
print('=== Signal-to-Noise Comparison ===')
print(f'{"":>20} {"EntangledHVec":>15} {"CliffordVec":>15}')
print(f'{"Noise floor":>20} {expected_jaccard:>15.6f} {E_abs_sim:>15.6f}')
print(f'{"Self-similarity":>20} {"1.0":>15} {"1.0":>15}')
print(f'{"SNR (1/noise)":>20} {1/expected_jaccard:>15.1f} {1/E_abs_sim:>15.1f}')

## 3. Empirical Results: Micro-Benchmarks

In [ ]:
micro = data['micro_benchmarks']

ops = [m['operation'] for m in micro]
e_ns = [m['entangled_ns'] for m in micro]
c_ns = [m['clifford_ns'] for m in micro]

x = np.arange(len(ops))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.bar(x - width/2, e_ns, width, label='EntangledHVec', color='#2196F3')
bars2 = ax1.bar(x + width/2, c_ns, width, label='CliffordVec', color='#FF5722')
ax1.set_ylabel('Latency (ns)')
ax1.set_title('Operation Latency: EntangledHVec vs CliffordVec')
ax1.set_xticks(x)
ax1.set_xticklabels(ops, rotation=30, ha='right')
ax1.legend()
ax1.set_yscale('log')

ratios = [m['ratio'] for m in micro]
colors = ['#4CAF50' if r <= 2 else '#FF9800' if r <= 10 else '#F44336' for r in ratios]
ax2.barh(ops, ratios, color=colors)
ax2.set_xlabel('Slowdown (Clifford / Entangled)')
ax2.set_title('CliffordVec Overhead Ratio')
ax2.axvline(x=1, color='black', linestyle='--', alpha=0.3)
ax2.axvline(x=10, color='red', linestyle='--', alpha=0.3, label='10x threshold')
ax2.legend()

plt.tight_layout()
plt.savefig('micro_benchmarks.png', bbox_inches='tight')
plt.show()

print('\nRaw numbers:')
for m in micro:
    print(f"  {m['operation']:<15} E={m['entangled_ns']:>8.1f}ns  C={m['clifford_ns']:>8.1f}ns  ratio={m['ratio']:.2f}x")

## 4. Similarity Distribution

In [ ]:
sim_dist = data['similarity_distribution']

fig, ax = plt.subplots(figsize=(10, 5))

labels = [d['representation'] for d in sim_dist]
metrics = ['self_sim_mean', 'random_pair_mean', 'bind_self_recovery']
metric_labels = ['Self-Similarity', 'Random Pair Mean', 'Bind-Unbind Recovery']

x = np.arange(len(metrics))
width = 0.3

for i, d in enumerate(sim_dist):
    values = [d[m] for m in metrics]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, values, width, label=d['representation'])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Similarity')
ax.set_title('Similarity Metric Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('similarity_distribution.png', bbox_inches='tight')
plt.show()

print('\nDetailed statistics:')
for d in sim_dist:
    print(f"\n  {d['representation']}:")
    print(f"    Self-sim:       {d['self_sim_mean']:.6f}")
    print(f"    Random pair:    {d['random_pair_mean']:.6f} +/- {d['random_pair_std']:.6f}")
    print(f"    Random range:   [{d['random_pair_min']:.6f}, {d['random_pair_max']:.6f}]")
    print(f"    Bind recovery:  {d['bind_self_recovery']:.6f}")

## 5. Query Scaling

In [ ]:
scaling = data['query_scaling']

ns = [s['n_patterns'] for s in scaling]
bf_us = [s['brute_force_query_us'] for s in scaling]
hf_us = [s['hopfield_query_us'] for s in scaling]
insert_ms = [s['insert_total_ms'] for s in scaling]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Query latency
axes[0].plot(ns, bf_us, 'o-', label='Brute Force', color='#2196F3', linewidth=2)
axes[0].plot(ns, hf_us, 's-', label='Hopfield', color='#FF5722', linewidth=2)
axes[0].set_xlabel('Number of Patterns')
axes[0].set_ylabel('Query Latency (us)')
axes[0].set_title('Query Latency vs Scale')
axes[0].legend()
axes[0].set_xscale('log')
axes[0].set_yscale('log')

# Hopfield slowdown factor
slowdowns = [s['hopfield_slowdown'] for s in scaling]
axes[1].bar(range(len(ns)), slowdowns, color='#FF9800')
axes[1].set_xticks(range(len(ns)))
axes[1].set_xticklabels([str(n) for n in ns])
axes[1].set_xlabel('Number of Patterns')
axes[1].set_ylabel('Slowdown (Hopfield / Brute Force)')
axes[1].set_title('Hopfield Overhead Factor')
axes[1].axhline(y=1, color='green', linestyle='--', alpha=0.5)

# Insert throughput
throughput = [n / (ms / 1000) for n, ms in zip(ns, insert_ms)]
axes[2].bar(range(len(ns)), throughput, color='#4CAF50')
axes[2].set_xticks(range(len(ns)))
axes[2].set_xticklabels([str(n) for n in ns])
axes[2].set_xlabel('Number of Patterns')
axes[2].set_ylabel('Insert Throughput (patterns/sec)')
axes[2].set_title('Insert Performance')

plt.tight_layout()
plt.savefig('query_scaling.png', bbox_inches='tight')
plt.show()

print('\nTop-1 agreement (brute-force vs Hopfield):')
for s in scaling:
    agree = 'AGREE' if s['top1_agreement'] else 'DISAGREE'
    print(f"  N={s['n_patterns']:<5}  BF={s['brute_top1_id']:<10}  HF={s['hopfield_top1_id']:<10}  {agree}")

## 6. Retrieval Quality

In [ ]:
quality = data['retrieval_quality']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Agreement rates
metrics_q = {
    'Top-1 Agreement': quality['top1_agreement_rate'],
    'Top-5 Jaccard': quality['top5_overlap_rate'],
}
bars = axes[0].bar(metrics_q.keys(), metrics_q.values(), color=['#2196F3', '#4CAF50'])
for bar, val in zip(bars, metrics_q.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.1%}', ha='center', va='bottom', fontsize=12)
axes[0].set_ylabel('Rate')
axes[0].set_title(f'Brute-Force vs Hopfield Agreement\n({quality["n_patterns"]} patterns, {quality["n_queries"]} queries)')
axes[0].set_ylim(0, 1.15)

# Hopfield sparsity characteristics
hopfield_metrics = {
    'Avg Results': quality['hopfield_avg_results'],
    'Sparse Rate\n(< 5 results)': quality['hopfield_sparsity_rate'],
}
colors = ['#FF5722', '#FF9800']
bars = axes[1].bar(hopfield_metrics.keys(), hopfield_metrics.values(), color=colors)
for bar, val in zip(bars, hopfield_metrics.values()):
    label = f'{val:.2f}' if val > 1 else f'{val:.1%}'
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 label, ha='center', va='bottom', fontsize=12)
axes[1].set_title('Hopfield Sparsity Characteristics')

plt.tight_layout()
plt.savefig('retrieval_quality.png', bbox_inches='tight')
plt.show()

## 7. Capacity & Crosstalk

In [ ]:
cap = data['capacity_estimation']

crosstalk_n = [c[0] for c in cap['measured_entangled_crosstalk']]
crosstalk_v = [c[1] for c in cap['measured_entangled_crosstalk']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Measured crosstalk vs N
ax1.plot(crosstalk_n, crosstalk_v, 'o-', color='#F44336', linewidth=2, markersize=8)
ax1.axhline(y=expected_jaccard, color='gray', linestyle='--', alpha=0.5,
            label=f'E[Jaccard] = {expected_jaccard:.4f}')

# Theoretical max Jaccard line
theory_n = np.logspace(1, 5, 100)
theory_max = [expected_jaccard + std_intersection/E_union * np.sqrt(2 * np.log(n))
              for n in theory_n]
ax1.plot(theory_n, theory_max, '--', color='#9C27B0', alpha=0.7, label='Theoretical E[max]')

ax1.set_xlabel('Number of Patterns (N)')
ax1.set_ylabel('Max Pairwise Similarity')
ax1.set_title('Crosstalk Growth with Pattern Count')
ax1.set_xscale('log')
ax1.legend()

# Capacity summary
summary_text = (
    f"EntangledHVec (Sparse Binary)\n"
    f"  Dimension: {cap['dim']}\n"
    f"  Active bits: {cap['active_bits']} (rho=1/256)\n"
    f"  Theoretical capacity: ~{cap['entangled_theoretical_capacity']:.0f}\n"
    f"\n"
    f"CliffordVec (Geometric Algebra)\n"
    f"  Cl({cap['clifford_n']},0)\n"
    f"  Algebra dimension: {cap['clifford_algebra_dim']}\n"
    f"  Sparse terms: {cap['clifford_terms']}\n"
    f"  Coefficient space: continuous\n"
    f"\n"
    f"Key insight: CliffordVec has lower noise floor\n"
    f"due to signed coefficients causing cancellation\n"
    f"in the inner product (random walk). Sparse binary\n"
    f"has strictly positive contributions (biased walk)."
)
ax2.text(0.1, 0.5, summary_text, transform=ax2.transAxes, fontsize=11,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Capacity Summary')

plt.tight_layout()
plt.savefig('capacity_analysis.png', bbox_inches='tight')
plt.show()

## 8. Codebook Design Space

Exploring the parameter space for optimal codebook configuration.

In [ ]:
# Parameter sweep: dimension vs sparsity vs capacity
dims = [1024, 4096, 8192, 16384, 32768, 65536]
rhos = [1/64, 1/128, 1/256, 1/512]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Noise floor vs dimension for different sparsities
for rho_val in rhos:
    noise = [rho_val / (2 - rho_val) for _ in dims]
    axes[0].plot(dims, noise, 'o-', label=f'rho=1/{int(1/rho_val)}')

axes[0].set_xlabel('Dimension D')
axes[0].set_ylabel('Expected Random Jaccard')
axes[0].set_title('Noise Floor vs Dimension')
axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log')
axes[0].legend()

# SNR vs dimension: signal is 1.0, noise is E[Jaccard]
# But more useful: max noise across N patterns
N_target = 10000
for rho_val in rhos:
    snrs = []
    for d in dims:
        k_val = int(d * rho_val)
        E_int = d * rho_val**2
        E_un = d * (2*rho_val - rho_val**2)
        var_int = d * rho_val**2 * (1 - rho_val**2)
        std_int = np.sqrt(var_int)
        max_noise = (E_int + std_int * np.sqrt(2 * np.log(N_target))) / E_un
        snr = 1.0 / max_noise
        snrs.append(snr)
    axes[1].plot(dims, snrs, 'o-', label=f'rho=1/{int(1/rho_val)}')

axes[1].set_xlabel('Dimension D')
axes[1].set_ylabel(f'SNR at N={N_target}')
axes[1].set_title(f'Signal-to-Noise Ratio (N={N_target} patterns)')
axes[1].set_xscale('log', base=2)
axes[1].axhline(y=10, color='green', linestyle='--', alpha=0.3, label='SNR=10 (good)')
axes[1].axhline(y=3, color='red', linestyle='--', alpha=0.3, label='SNR=3 (marginal)')
axes[1].legend()

plt.tight_layout()
plt.savefig('codebook_design_space.png', bbox_inches='tight')
plt.show()

In [ ]:
# Clifford vs Entangled: theoretical SNR comparison
# CliffordVec noise: random walk with k^2/D shared blades, +/-1 contributions
# noise ~ sqrt(k^2/D) / k = 1/sqrt(D)
# EntangledHVec noise: biased walk with rho^2*D shared bits
# noise ~ rho / (2 - rho)

fig, ax = plt.subplots(figsize=(10, 6))

dims_fine = np.logspace(8, 18, 100, base=2)

for rho_val in [1/128, 1/256, 1/512]:
    entangled_noise = rho_val / (2 - rho_val) * np.ones_like(dims_fine)
    ax.plot(dims_fine, entangled_noise, '--',
            label=f'Entangled rho=1/{int(1/rho_val)}', alpha=0.7)

# CliffordVec noise floor: E[|sim|] ~ sqrt(k^2/D) / k = 1/sqrt(D)
# where k = D/256 (matching EntangledHVec active count)
clifford_noise = 1.0 / np.sqrt(dims_fine)
ax.plot(dims_fine, clifford_noise, 'r-', linewidth=2.5,
        label='CliffordVec (any rho)', alpha=0.9)

ax.set_xlabel('Dimension D')
ax.set_ylabel('Noise Floor (Expected Random Similarity)')
ax.set_title('Noise Floor: EntangledHVec vs CliffordVec')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('noise_floor_comparison.png', bbox_inches='tight')
plt.show()

print('\nKey finding:')
print(f'  At D=16384:')
print(f'    EntangledHVec noise (rho=1/256): {1/256 / (2 - 1/256):.6f}')
print(f'    CliffordVec noise:               {1/np.sqrt(16384):.6f}')
print(f'    CliffordVec advantage:            {(1/256/(2-1/256)) / (1/np.sqrt(16384)):.1f}x lower noise')

## 9. Summary & Recommendations

In [ ]:
print('='*70)
print('HMS CODEBOOK & BENCHMARK SUMMARY')
print('='*70)
print()
print('MICRO-BENCHMARKS:')
for m in micro:
    verdict = 'OK' if m['ratio'] < 10 else 'SLOW'
    print(f"  {m['operation']:<15} Clifford is {m['ratio']:.1f}x slower [{verdict}]")
print()
print('SIMILARITY PROPERTIES:')
for d in sim_dist:
    print(f"  {d['representation']}:")
    print(f"    Noise floor: {d['random_pair_mean']:.6f}")
    print(f"    Bind recovery: {d['bind_self_recovery']:.4f}")
print()
print('QUERY SCALING:')
for s in scaling:
    print(f"  N={s['n_patterns']:<5}  BF={s['brute_force_query_us']:>8.1f}us  "
          f"HF={s['hopfield_query_us']:>8.1f}us  agree={s['top1_agreement']}")
print()
print('RECOMMENDATIONS:')
print('  1. EntangledHVec remains the production workhorse (fast, proven)')
print('  2. CliffordVec bind (geometric product) is expensive due to O(k^2)')
print('     quadratic term expansion -- acceptable for offline/batch')
print('  3. CliffordVec has theoretically lower noise floor (signed coefficients)')
print('  4. Hopfield retrieval: quality comparable to brute-force, but slower')
print('     at current scale. Advantage grows with pattern count (sparse output)')
print('  5. Next steps: optimize geometric product sparsity, wire Hopfield')
print('     into query router for auto-selection based on pattern count')